# bloch4spin Demo: Spin-j Evolution and Sequential Measurement

This notebook demonstrates:

1. Construction of spin operator matrices (Jx, Jy, Jz, J_+, J_-) for a finite spin system with dimension d = 2j + 1.
2. Building the Bloch-space Liouvillian from a Hamiltonian H = Jy.
3. Projective measurements of Jz at multiple times using Bloch-space measurement superoperators.
4. Computing joint probability tensors via the experiment engine and validating against a direct theoretical transition model T = |U|^2.

Key identities:
- Bloch normalization: T_0^{(0)} = I / sqrt(d)
- Probability extraction for an (unnormalized) post-measurement Bloch vector r': p = sqrt(d) * r'_0

We test for d = 2 (spin-1/2) and d = 3 (spin-1) over two and three measurement times.

### Initialization

In [1]:
import numpy as np
from numpy.typing import NDArray

from bloch4spin.basis import bloch_init
from bloch4spin.evolution import GeneralizedBlochEvolutionMatrix, GeneralizedBlochState
from bloch4spin.observable import GeneralizedBlochObservable
from bloch4spin.engine import run

### Spin operators and projectors onto Jz eigenstates

In [2]:
def spin_ops(d: int):
    """Return (Jx, Jy, Jz, J_plus, J_minus) for spin j with d=2j+1 in the |m_z> basis.
    Basis ordering: index i in 0..d-1 corresponds to m = j - i (descending m_z).
    Ladder actions:
      J_+ |m> = sqrt((j-m)(j+m+1)) |m+1>
      J_- |m> = sqrt((j+m)(j-m+1)) |m-1>
    """
    j = (d - 1) / 2.0
    Jp = np.zeros((d, d), dtype=complex)
    Jm = np.zeros((d, d), dtype=complex)
    # Build J_+
    for i in range(1, d):  # i corresponds to m = j - i
        amp = np.sqrt(i * (d - i))
        Jp[i-1, i] = amp
    # Build J_- (Hermitian adjoint of J_+)
    for i in range(0, d-1):
        amp = np.sqrt((i + 1) * (d - (i + 1)))
        Jm[i+1, i] = amp
    Jx = 0.5 * (Jp + Jm)
    Jy = (Jp - Jm) / (2j)  # Python literal 2j == 2*i; division by (2i)
    m_vals = np.array([j - i for i in range(d)], dtype=float)
    Jz = np.diag(m_vals)
    return Jx, Jy, Jz, Jp, Jm

def projectors_Jz(d: int):
    Ps = []
    for i in range(d):
        P = np.zeros((d, d), dtype=complex)
        P[i, i] = 1.0
        Ps.append(P)
    return Ps

### Transition matrix & theoretical solution

In [3]:
def transition_matrix_from_Jy(Jy: NDArray[np.complex128], delta_t: float) -> NDArray[np.float64]:
    """Return stochastic transition matrix T = |U|^2 for evolution time delta_t under H = Jy.
    U = V exp(-i w delta_t) V^† where Jy = V diag(w) V^†.
    Row index: final m; column index: initial m.
    """
    w, V = np.linalg.eigh(Jy)
    U = (V * np.exp(-1j * w * delta_t)) @ V.conj().T
    T = np.abs(U) ** 2
    return T.real

def theoretical_joint(d: int, times: list[float]) -> NDArray[np.float64]:
    """Joint distribution for sequential Jz measurements at specified times
    starting from maximally mixed state and evolving under H = Jy.
    First measurement is uniform; subsequent steps use chained transitions.
    Shape: (d, d, ..., d) with one axis per measurement.
    """
    _, Jy, _, _, _ = spin_ops(d)
    M = len(times)
    if M == 0:
        return np.array([1.0])
    joint = np.full((d,), 1.0 / d, dtype=float)
    for k in range(1, M):
        delta_t = float(times[k] - times[k - 1])
        T = transition_matrix_from_Jy(Jy, delta_t)
        joint = np.einsum('...m,nm->...mn', joint, T)
    return joint

### Engine comparison helper 

In [4]:
def run_case(d: int, times: list[float], *, atol=1e-10):
    """Run engine vs theory for given dimension and measurement times.
    Steps:
      1. Initialize ITO-based Bloch basis.
      2. Build Hamiltonian H = Jy and its Liouvillian.
      3. Form maximally mixed initial state.
      4. Construct Jz projective measurement superoperators.
      5. Execute experiment schedule with engine.run.
      6. Compare joint probabilities to theoretical chained transition model.
    """
    print(f"\n=== Test d={d}, times={times} ===")
    bloch_init(d)
    _, Jy, _, _, _ = spin_ops(d)
    L = GeneralizedBlochEvolutionMatrix.from_Hamiltonian(Jy)
    rho0 = np.eye(d, dtype=complex) / d
    r0 = GeneralizedBlochState.from_matrix(rho0)
    obs_list = [GeneralizedBlochObservable.from_projector(P) for P in projectors_Jz(d)]
    schedule = [(float(t), obs_list) for t in times]
    joint_engine = run(L, r0, schedule, norm_mode="normalized")
    joint_theory = theoretical_joint(d, times)
    print("engine joint shape:", joint_engine.shape)
    print("theory joint shape:", joint_theory.shape)
    print("engine joint:\n", np.round(joint_engine, 6))
    print("theory joint:\n", np.round(joint_theory, 6))
    err = np.max(np.abs(joint_engine - joint_theory))
    print(f"max |diff| = {err:.3e}")
    assert err < atol, f"Mismatch exceeds tolerance: {err} >= {atol}" 
    print("PASS")

### Execute test scenarios 

In [5]:
_times_two = [0.7, 1.9]
_times_three = [0.3, 1.1, 2.0]

for _d in (2, 3):
    run_case(_d, _times_two)
    run_case(_d, _times_three)


=== Test d=2, times=[0.7, 1.9] ===
engine joint shape: (2, 2)
theory joint shape: (2, 2)
engine joint:
 [[0.340589 0.159411]
 [0.159411 0.340589]]
theory joint:
 [[0.340589 0.159411]
 [0.159411 0.340589]]
max |diff| = 2.220e-16
PASS

=== Test d=2, times=[0.3, 1.1, 2.0] ===
engine joint shape: (2, 2, 2)
theory joint shape: (2, 2, 2)
engine joint:
 [[[0.343925 0.080252]
  [0.014345 0.061478]]

 [[0.061478 0.014345]
  [0.080252 0.343925]]]
theory joint:
 [[[0.343925 0.080252]
  [0.014345 0.061478]]

 [[0.061478 0.014345]
  [0.080252 0.343925]]]
max |diff| = 2.776e-16
PASS

=== Test d=3, times=[0.7, 1.9] ===
engine joint shape: (3, 3)
theory joint shape: (3, 3)
engine joint:
 [[0.154668 0.144783 0.033882]
 [0.144783 0.043768 0.144783]
 [0.033882 0.144783 0.154668]]
theory joint:
 [[0.154668 0.144783 0.033882]
 [0.144783 0.043768 0.144783]
 [0.033882 0.144783 0.154668]]
max |diff| = 1.943e-16
PASS

=== Test d=3, times=[0.3, 1.1, 2.0] ===
engine joint shape: (3, 3, 3)
theory joint shape: (3

## Conclusion

The engine-produced joint probability tensors match the analytical chained-transition construction to numerical precision (max |diff| < 1e-10 for tested cases). This validates:

- Correct Hamiltonian → Liouvillian mapping for H = Jy.
- Probability extraction via r'_0 → p.
- Measurement branching and ordering in `engine.run`.

You can adapt this template for dissipative dynamics (adding Lindblad terms to the Liouvillian) or extended POVM measurements by replacing projector construction with `from_POVM_elem` or `from_Kraus` constructors.